# Week 3 — Context Engineering II (Local with LiteLLM & Ollama)

**AI Agentic Engineering · Corte 1**

Companion notebook to `week-03-context-engineering-ii-content.html`.

**You will practice:**
1. Sliding-window truncation vs. rolling summarization on a synthetic conversation.
2. Generating embeddings and computing cosine similarity locally.
3. A minimal semantic search function (the seed of next week's RAG system).
4. A light ADK / LangChain preview of the summarization call.
5. Two open exercises.

**Environment:** Running 100% locally with Ollama, RTX GPU acceleration, LiteLLM (`qwen2.5:14b` & `nomic-embed-text`).

In [1]:
%pip install -q --upgrade litellm google-adk langchain-community langchain-ollama python-dotenv numpy pydantic

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import numpy as np
from dotenv import load_dotenv
import litellm
from litellm import completion
from pydantic import BaseModel

load_dotenv()
MODEL = "ollama_chat/qwen2.5:14b"
EMBED_MODEL = "ollama/nomic-embed-text" 

## 1. Sliding window vs. rolling summarization

In [7]:
history = [
    {"role": "user", "content": "Hi, I'm planning a trip to Peru."},
    {"role": "assistant", "content": "Great! When are you thinking of traveling, and for how long?"},
    {"role": "user", "content": "Sometime in August, for about 2 weeks."},
    {"role": "assistant", "content": "August is dry season in the Andes, good for Machu Picchu. Any budget in mind?"},
    {"role": "user", "content": "Around $2000 total, excluding flights."},
    {"role": "assistant", "content": "That's workable for hostels and local transport. Interested in the Amazon too, or just the highlands?"},
    {"role": "user", "content": "Just the highlands. I don't do well with humidity."},
    {"role": "assistant", "content": "Noted — highlands only. Cusco, Sacred Valley, and Machu Picchu it is."},
    {"role": "user", "content": "Also I'm vegetarian, will that be an issue?"},
    {"role": "assistant", "content": "Not at all, Peruvian highland cuisine has plenty of vegetarian options."},
]

def sliding_window(history, max_turns=4):
    return history[-max_turns:]

print("--- Sliding window (last 4) ---")
for m in sliding_window(history, max_turns=4):
    print(m["role"], ":", m["content"])

--- Sliding window (last 4) ---
user : Just the highlands. I don't do well with humidity.
assistant : Noted — highlands only. Cusco, Sacred Valley, and Machu Picchu it is.
user : Also I'm vegetarian, will that be an issue?
assistant : Not at all, Peruvian highland cuisine has plenty of vegetarian options.


In [4]:
def summarize_history(history):
    transcript = "\n".join(f"{m['role']}: {m['content']}" for m in history)
    prompt = (
        "Summarize the key facts, decisions, and constraints from this conversation "
        "in 3-5 bullet points. Be specific.\n\n" + transcript
    )
    response = completion(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
    )
    return response.choices[0].message.content

def compact_context(history, keep_recent=4):
    if len(history) <= keep_recent:
        return history
    older, recent = history[:-keep_recent], history[-keep_recent:]
    summary = summarize_history(older)
    return [{"role": "system", "content": f"Conversation summary so far:\n{summary}"}] + recent

compacted = compact_context(history, keep_recent=4)
print("--- Rolling summarization + last 4 ---")
for m in compacted:
    print(m["role"], ":", m["content"])

--- Rolling summarization + last 4 ---
system : Conversation summary so far:
- Travel dates: August, for a duration of 2 weeks.
- Budget: $2000 total, excluding airfare.
- Destination focus: Considering both the highlands (Andes region) and possibly the Amazon rainforest.
- Seasonal information: August is in the dry season, ideal for visiting Machu Picchu.
user : Just the highlands. I don't do well with humidity.
assistant : Noted — highlands only. Cusco, Sacred Valley, and Machu Picchu it is.
user : Also I'm vegetarian, will that be an issue?
assistant : Not at all, Peruvian highland cuisine has plenty of vegetarian options.


Compare the two outputs above. Sliding window silently drops "vegetarian" and "highlands only, no humidity"
if the window is small enough — summarization keeps them. Try lowering `max_turns` to 2 and see which strategy
still remembers the vegetarian constraint.

## 2. Embeddings and cosine similarity

In [4]:
def embed(text):
    result = litellm.embedding(model=EMBED_MODEL, input=text)
    return result.data[0]["embedding"]

def cosine_similarity(a, b):
    a, b = np.array(a), np.array(b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

v_brake = embed("The bike's front brake feels loose.")
v_caliper = embed("The caliper on the front wheel isn't gripping properly.")
v_weather = embed("It might rain this weekend.")

print("brake vs caliper (related): ", round(cosine_similarity(v_brake, v_caliper), 3))
print("brake vs weather (unrelated):", round(cosine_similarity(v_brake, v_weather), 3))

brake vs caliper (related):  0.737
brake vs weather (unrelated): 0.463


## 3. A minimal semantic search function

In [5]:
candidates = [
    "To fix a loose brake, tighten the caliper bolts and check the brake pads for wear.",
    "Flat tires are usually caused by punctures or worn-out inner tubes.",
    "Our rental bikes come in small, medium, and large frame sizes.",
    "Helmets are provided free of charge with every rental.",
    "The gear shifter may need cable tension adjustment if shifting feels sluggish.",
    "Rentals can be extended by messaging support at least 2 hours before the due time.",
]
candidate_vectors = [embed(c) for c in candidates]

def semantic_search(query, top_k=3):
    q_vec = embed(query)
    scored = [(cosine_similarity(q_vec, v), text) for v, text in zip(candidate_vectors, candidates)]
    scored.sort(reverse=True)
    return scored[:top_k]

for score, text in semantic_search("my brake feels wobbly and doesn't stop well"):
    print(f"{score:.3f}  {text}")

0.708  To fix a loose brake, tighten the caliper bolts and check the brake pads for wear.
0.615  The gear shifter may need cable tension adjustment if shifting feels sluggish.
0.544  Flat tires are usually caused by punctures or worn-out inner tubes.


Notice the top result shares almost no exact words with the query ("wobbly", "doesn't stop well" vs.
"loose", "caliper bolts") — that's semantic search working as intended. Next week we scale this exact pattern
into a full RAG pipeline with chunking and a real vector database.

## 4. Light preview: the summarization call via ADK and LangChain

Same idea as previous weeks — reproducing one call (this time, `summarize_history`) through different tools.

In [8]:
from google.adk.agents import Agent
from google.adk.models.lite_llm import LiteLlm
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types

summarizer_agent = Agent(
    model=LiteLlm(model=MODEL),
    name="summarizer_agent",
    instruction="Summarize the given conversation in 3-5 specific bullet points.",
)

async def ask_adk_agent(agent, prompt, app_name="week3_app", user_id="student"):
    session_service = InMemorySessionService()
    runner = Runner(
        agent=agent,
        app_name=app_name,
        session_service=session_service,
        auto_create_session=True,
    )
    session = await session_service.create_session(app_name=app_name, user_id=user_id)
    content = types.Content(role="user", parts=[types.Part.from_text(text=prompt)])
    final_text = ""
    async for event in runner.run_async(user_id=user_id, session_id=session.id, new_message=content):
        if event.content and event.content.parts:
            for p in event.content.parts:
                if p.text:
                    final_text += p.text
    return final_text

transcript = "\n".join(f"{m['role']}: {m['content']}" for m in history[:-4])
adk_summary = await ask_adk_agent(summarizer_agent, transcript)
print("ADK summary:\n", adk_summary)

ADK summary:
 - The user is planning a trip to Peru in August for two weeks.
- The assistant recommends the dry season in August as ideal for visiting Machu Picchu.
- The user has a budget of approximately $2000, excluding flight costs.
- The assistant suggests that the budget is suitable for hostels and local transportation.
- The assistant inquires if the user is interested in exploring the Amazon region or just the highlands.


In [9]:
from langchain_ollama import ChatOllama, OllamaEmbeddings

ollama_model_name = MODEL.replace("ollama_chat/", "").replace("ollama/", "")
llm = ChatOllama(model=ollama_model_name)
lc_summary = llm.invoke(
    "Summarize the key facts, decisions, and constraints from this conversation in 3-5 bullet points:\n\n"
    + transcript
)
print("LangChain summary:\n", lc_summary.content)

# LangChain also wraps the embedding model:
embed_model_name = EMBED_MODEL.replace("ollama/", "")
lc_embeddings = OllamaEmbeddings(model=embed_model_name)
lc_vector = lc_embeddings.embed_query("The bike's front brake feels loose.")
print("\nLangChain embedding dims:", len(lc_vector))

LangChain summary:
 - Travel dates: August, for approximately 2 weeks.
- Budget: $2000 total, excluding flights.
- Primary destination considerations: Dry season in the Andes, suitable for visiting Machu Picchu.
- Accommodation and transportation: Budget allows for hostels and local transport.
- Areas of interest: User is considering both the Amazon and the highlands but hasn't decided yet.

LangChain embedding dims: 768


## 5. Exercises

In [10]:
# TODO Exercise 1 — Ranked semantic search with scores
# Extend `semantic_search` (or write a new version) so it also accepts a `threshold` parameter
# and drops any result below that similarity score, in addition to returning top_k.
# Test it with a query that has no good match in `candidates` and confirm it returns nothing (or few) results.

def semantic_search_with_threshold(query: str, top_k: int = 3, threshold: float = 0.5):
    q_vec = embed(query)
    scored = [(cosine_similarity(q_vec, v), text) for v, text in zip(candidate_vectors, candidates)]
    scored.sort(reverse=True)
    filtered = [(score, text) for score, text in scored if score >= threshold]
    return filtered[:top_k]

print("--- Query with match (threshold=0.45) ---")
for score, text in semantic_search_with_threshold("my brake feels loose and slippery", top_k=3, threshold=0.6):
    print(f"{score:.3f}  {text}")

print("\n--- Query without match (threshold=0.55) ---")
results_unrelated = semantic_search_with_threshold("worst foods around the world", top_k=3, threshold=0.45)
if not results_unrelated:
    print("No relevant results found above threshold.")
else:
    for score, text in results_unrelated:
        print(f"{score:.3f}  {text}")

--- Query with match (threshold=0.45) ---
0.745  To fix a loose brake, tighten the caliper bolts and check the brake pads for wear.
0.610  The gear shifter may need cable tension adjustment if shifting feels sluggish.

--- Query without match (threshold=0.55) ---
No relevant results found above threshold.


In [8]:
# TODO Exercise 2 — Structured memory extraction
# Define a Pydantic model `TripMemory` with fields like `destination`, `month`, `budget_usd`,
# `dietary_restrictions: list[str]`, `regions_of_interest: list[str]`.
# Use schema-forced JSON output (from Week 2) to extract a TripMemory instance from the
# `history` conversation above, instead of a paragraph summary.

class TripMemory(BaseModel):
    destination: str
    month: str
    budget_usd: float
    dietary_restrictions: list[str]
    regions_of_interest: list[str]

transcript_full = "\n".join(f"{m['role']}: {m['content']}" for m in history)

response = completion(
    model=MODEL,
    messages=[
        {
            "role": "user",
            "content": f"Extract structured trip memory from this conversation:\n\n{transcript_full}",
        }
    ],
    response_format=TripMemory,
)

trip_memory = TripMemory.model_validate_json(response.choices[0].message.content)
print("Extracted TripMemory:")
print(trip_memory)
print(f"Destination: {trip_memory.destination}")
print(f"Month: {trip_memory.month}")
print(f"Budget: ${trip_memory.budget_usd}")
print(f"Dietary: {trip_memory.dietary_restrictions}")
print(f"Regions: {trip_memory.regions_of_interest}")

Extracted TripMemory:
destination='Peru' month='August' budget_usd=2000.0 dietary_restrictions=['vegetarian'] regions_of_interest=['Cusco', 'Sacred Valley', 'Machu Picchu', 'highlands only', 'excluding Amazon due to humidity']
Destination: Peru
Month: August
Budget: $2000.0
Dietary: ['vegetarian']
Regions: ['Cusco', 'Sacred Valley', 'Machu Picchu', 'highlands only', 'excluding Amazon due to humidity']


## Next week

Week 4 — **Retrieval-Augmented Generation (RAG)**: chunking strategies, ChromaDB and FAISS, and building a
complete, functional RAG system — the foundation of your Corte 1 project. See `week-04-rag-content.html`.